In [ ]:
# NOTEBOOK NAME
# MassFeatureStatTimeChunks.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import math
import calendar

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *

from pyproj import Geod

In [ ]:
def AddSparseIndexedVars(ds):
    """
    For every variable with only the (tracks) dimension, create a new
    variable with the suffix '_sparse_indexed' and dimension (sparse_index),
    by mapping through tracks_indices.

    Parameters
    ----------
    ds : xr.Dataset

    Returns
    -------
    xr.Dataset with additional '_sparse_indexed' variables
    """
    ds_new        = ds.copy()
    track_indices = ds['tracks_indices'].values.astype(int)

    for var in ds.data_vars:
        if ds[var].dims == ('tracks',):
            ds_new[f'{var}_sparse_indexed'] = xr.DataArray(
                ds[var].values[track_indices],
                dims='sparse_index'
            )

    return ds_new

In [ ]:
# LOAD IN MASS FEATURE STATS NETCDF (IN SPARSE FORMAT)

StartFileDateStr = '20240201'
EndFileDateStr = '20240229'
RadarIDno = 22
QualityControlOption = 2

SparseStoragePath = (f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/QC{QualityControlOption}/V5/Period_{StartFileDateStr}_{EndFileDateStr}/trackstats_sparse_{StartFileDateStr}.000000_{EndFileDateStr}.235500.nc')

SparseFXR = xr.open_dataset(SparseStoragePath)

SparseFXR = AddSparseIndexedVars(SparseFXR)

# FeatureXR = AddPropagationVars(FeatureXR)

# FeatureXR = AddFrameTimeVars(FeatureXR)


In [ ]:
# add to the sparse data set a time variable that lists the time rounded down to 5-mins AS IN THE NAME OF THE RADAR SCAN

SparseFXR['base_time_fivemin'] = xr.DataArray(
    pd.DatetimeIndex(SparseFXR['base_time'].values).floor('5min').to_numpy(),
    dims='sparse_index'
)

# add to the sparse data set an index time variable that lists the number of 5-mins scans into the period
t0 = SparseFXR['base_time_fivemin'].values[0]

SparseFXR['base_time_index'] = xr.DataArray(
    ((SparseFXR['base_time_fivemin'].values - t0) / np.timedelta64(5, 'm')).astype(int),
    dims='sparse_index'
)

In [ ]:
SparseFXR

In [ ]:
RangeStartHour = 4
RangeEndHour = 6

MinDuration = 12 # [minutes]

# minimum number of frames required to meet time requirement
MinFrames = int(np.ceil(MinDuration/5) + 1)

BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)

FeatureFrames = SparseFXR['times_indices'].values
TrackDurations = SparseFXR['track_duration_sparse_indexed'].values



TimeMask  = (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartHour*60) & (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute < RangeEndHour*60)

BirthMask = (FeatureFrames == 0)
DeathMask = (FeatureFrames == TrackDurations-1)

DurationMask = (TrackDurations >= MinFrames)

BirthComboMask = TimeMask * DurationMask * BirthMask
DeathComboMask = TimeMask * DurationMask * DeathMask
AllComboMask   = TimeMask * DurationMask


In [ ]:
for i in range(0,100):

    print('sparse index: ' + str(SparseFXR['sparse_index'].values[i]))
    
    print('track: ' + str(SparseFXR['tracks_indices'].values[i]))
    
    print('frame: ' + str(SparseFXR['times_indices'].values[i]))
    
    print('time: ' + str(SparseFXR['base_time'].values[i]))

    print('')



In [ ]:
# CHAD PLOT
# SPARSE LOADED VERSION
# FEATURE LOCATIONS (BIRTH DEATH AND ALL) AND KERENEL DENSITY

def PlotFeatureLocationsSparse(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    TimeRange,
    DotsIncluded,
    MinDuration,
):
    """
    Plots feature locations (birth, death, or all frames) from a pre-loaded
    sparse PyFLEXTRKR xarray Dataset (SparseFXR).

    Produces two plots:
        1. Scatter plot  — small red dots on terrain background
        2. Density plot  — kernel density estimate on terrain background

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string (e.g. '22')
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD' inclusive date range
    TimeRange            : str        - 'HH:MM-HH:MM' UTC time-of-day window
    DotsIncluded         : str        - 'Birth', 'Death', or 'All'
    MinDuration          : float      - Minimum feature duration in minutes
    RangeStartHour       : int        - Start hour of time window (UTC)
    RangeEndHour         : int        - End hour of time window (UTC)
    """

    import math
    import pandas as pd
    from scipy.stats           import gaussian_kde
    from pathlib               import Path
    from matplotlib.colors     import ListedColormap, BoundaryNorm
    import matplotlib.ticker   as mticker
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    import cartopy.crs         as ccrs
    import cartopy.feature     as cfeature

    # ── Parse time range ───────────────────────────────────────────────────────

    time_start_str, time_end_str = TimeRange.split('-')
    RangeStartHour  = int(time_start_str.split(':')[0])
    RangeStartMin   = int(time_start_str.split(':')[1])
    RangeEndHour    = int(time_end_str.split(':')[0])
    RangeEndMin     = int(time_end_str.split(':')[1])
    
    RangeStartMins  = RangeStartHour * 60 + RangeStartMin
    RangeEndMins    = RangeEndHour   * 60 + RangeEndMin

    # ── Input validation ───────────────────────────────────────────────────────

    if DotsIncluded not in ('Birth', 'Death', 'All'):
        raise ValueError("DotsIncluded must be 'Birth', 'Death', or 'All'.")

    # ── Parse date range for titles / save paths ───────────────────────────────

    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    # ── Build masks ────────────────────────────────────────────────────────────

    MinFrames      = int(np.ceil(MinDuration / 5) + 1)

    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values

    TimeMask = (
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartMins) &
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute <  RangeEndMins)
    )


    BirthMask    = (FeatureFrames == 0)
    DeathMask    = (FeatureFrames == TrackDurations - 1)
    DurationMask = (TrackDurations >= MinFrames)

    BirthComboMask = TimeMask & DurationMask & BirthMask
    DeathComboMask = TimeMask & DurationMask & DeathMask
    AllComboMask   = TimeMask & DurationMask

    # ── Select lat/lon based on DotsIncluded ──────────────────────────────────

    if DotsIncluded == 'Birth':
        ActiveMask = BirthComboMask
    elif DotsIncluded == 'Death':
        ActiveMask = DeathComboMask
    else:  # 'All'
        ActiveMask = AllComboMask

    all_lats = SparseFXR['core_meanlat'].values[ActiveMask]
    all_lons = SparseFXR['core_meanlon'].values[ActiveMask]

    # ── Drop any NaN lat/lon ───────────────────────────────────────────────────

    valid = np.isfinite(all_lats) & np.isfinite(all_lons)
    all_lats = all_lats[valid]
    all_lons = all_lons[valid]

    print(f"Duration filter : >= {MinDuration} min  →  >= {MinFrames} frames")
    print(f"Date range      : {date_start.date()} to {date_end.date()}")
    print(f"Time window     : {TimeRange} UTC")
    print(f"Mode            : {DotsIncluded}")
    print(f"Points collected: {len(all_lats)}")

    if len(all_lats) == 0:
        print("No valid points found — nothing to plot.")
        return

    # ── Shared map extent ──────────────────────────────────────────────────────

    lon_min = float(np.nanmin(all_lons))
    lon_max = float(np.nanmax(all_lons))
    lat_min = float(np.nanmin(all_lats))
    lat_max = float(np.nanmax(all_lats))

    # ── Shared title suffix ────────────────────────────────────────────────────

    title_suffix = (
        f'{RadarSiteName} Radar  |  {DotsIncluded} Locations\n'
        f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
        f'{TimeRange} UTC  |  Min Duration: {MinDuration} min  |  '
        f'({len(all_lats)} points)'
    )

    # ── Save folder ────────────────────────────────────────────────────────────

    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_{TimeRange.replace(":", "").replace("-", "_")}_'
        f'{DotsIncluded}_{int(MinDuration)}minMin'
    )

        # ── Load first radar file to get radar location ────────────────────────────
    
    RadarSiteName, LonShift = GrabRadarInfo(RadarIDno)
    
    first_date_str  = DateRange[:8]
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{first_date_str}/QC2/'
        f'{RadarIDno}_{first_date_str}_000000_QC2.nc'
    )
    
    # Fall back to first available file if 000000 is missing
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {first_date_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')
    
    RadarXR_ref    = xr.open_dataset(radar_file_path)
    RadarLon       = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat       = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    
    print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')


    # ── Helper: build base map ─────────────────────────────────────────────────

    def build_basemap():

        fig, ax = plt.subplots(
            figsize    = (8, 6),
            subplot_kw = {'projection': ccrs.PlateCarree()},
            facecolor  = 'white',
        )

        # --- Terrain background ---
        try:
            gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
            dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

            if dem_da is None:
                raise ValueError('GEBCO DEM returned None')

            dem_lon  = dem_da.lon.values
            dem_lat  = dem_da.lat.values
            dem_data = dem_da.values

            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            if not np.any(np.isfinite(dem_data)):
                raise ValueError('DEM has no finite values in this domain')

            colours_topo = [
                '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
                '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
            ]
            bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
            cmap_elev   = ListedColormap(colours_topo)
            norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

            ax.pcolormesh(
                dem_lon_2d, dem_lat_2d, dem_data,
                cmap      = cmap_elev,
                norm      = norm_topo,
                alpha     = 1.0,
                transform = ccrs.PlateCarree(),
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels    = [0.0],
                colors    = 'black',
                linewidths= 0.5,
                transform = ccrs.PlateCarree(),
                zorder    = 15,
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels    = [400.0],
                colors    = 'black',
                linewidths= 0.3,
                transform = ccrs.PlateCarree(),
                zorder    = 15,
            )
            print('  Terrain shading loaded successfully.')

        except Exception as e:
            print(f'  Terrain shading failed: {e}')
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
            ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

        # --- Gridlines ---
        gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
        gl_minor.xlocator = mticker.MultipleLocator(0.1)
        gl_minor.ylocator = mticker.MultipleLocator(0.1)

        gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
        gl_mid.xlocator = mticker.MultipleLocator(0.5)
        gl_mid.ylocator = mticker.MultipleLocator(0.5)

        gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
        gl_major.xlocator = mticker.MultipleLocator(1.0)
        gl_major.ylocator = mticker.MultipleLocator(1.0)

        for gl in (gl_mid, gl_major):
            gl.xformatter   = LONGITUDE_FORMATTER
            gl.yformatter   = LATITUDE_FORMATTER
            gl.top_labels   = False
            gl.right_labels = False

        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

            # --- Radar location star ---
        ax.plot(RadarLon, RadarLat,
                marker='*', color='black', markersize=8,
                transform=ccrs.PlateCarree(), zorder=30)
        ax.plot(RadarLon, RadarLat,
                marker='*', color='white', markersize=4,
                transform=ccrs.PlateCarree(), zorder=31)

        return fig, ax

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 1 — Scatter (red dots)
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating scatter plot...')
    fig1, ax1 = build_basemap()

    ax1.scatter(
        all_lons, all_lats,
        s          = 6,
        color      = 'red',
        alpha      = 0.6,
        transform  = ccrs.PlateCarree(),
        zorder     = 25,
        edgecolors = 'none',
    )

    ax1.set_title(f'Feature {title_suffix}', fontsize=10)

    plt.tight_layout()
    SavePath1 = SaveFolder + SaveBase + '_Scatter.png'
    # plt.savefig(SavePath1, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Scatter plot saved to: {SavePath1}')

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 2 — Kernel Density
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating density plot...')
    fig2, ax2 = build_basemap()

    xy          = np.vstack([all_lons, all_lats])
    kde         = gaussian_kde(xy, bw_method=0.1)

    lon_grid    = np.linspace(lon_min, lon_max, 300)
    lat_grid    = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    grid_coords = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])

    kde_values  = kde(grid_coords).reshape(lon_mesh.shape)

    # --- Convert to points per 100 km² ---
    mean_lat       = np.mean(all_lats)
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(mean_lat))
    km2_per_deg2   = km_per_deg_lat * km_per_deg_lon
    n_points       = len(all_lats)
    kde_per_100km2 = (kde_values / km2_per_deg2) * n_points * 100.0

    # --- Dynamic colour scale ---
    kde_max        = float(np.nanmax(kde_per_100km2))
    level_step     = kde_max / 25.0
    levels_fill    = np.arange(0, kde_max + level_step, level_step)
    levels_visible = levels_fill[1:]

    kde_fill = ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_fill,
        cmap      = 'jet',
        alpha     = 0.0,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_visible,
        cmap      = 'jet',
        alpha     = 0.25,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    # --- Colourbar ---
    cax = fig2.add_axes([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    sm = plt.cm.ScalarMappable(
        cmap = 'jet',
        norm = plt.Normalize(vmin=0, vmax=kde_max),
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Locations per 100 km²', fontsize=9)

    tick_vals = np.linspace(0, kde_max, 11)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=8)

    ax2.set_title(f'Feature Density {title_suffix}', fontsize=10)

    plt.tight_layout()
    plt.draw()

    cax.set_position([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    SavePath2 = SaveFolder + SaveBase + '_Density.png'
    # plt.savefig(SavePath2, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Density plot saved to: {SavePath2}')


In [ ]:
PlotFeatureLocationsSparse(
    SparseFXR            = SparseFXR,
    RadarIDno            = '41',
    RadarSiteName        = 'Willis Island',
    QualityControlOption = 2,
    DateRange            = '20240201-20240229',
    TimeRange            = '04:00-06:00',
    DotsIncluded         = 'All',
    MinDuration          = 30,
)



In [ ]:
# MANY CHAD FUNCTIONS
# USED TO CREATE GRID OF KERNEL DENSITY PLOTS

def _BuildFeatureMasks(SparseFXR, TimeRange, MinDuration):
    """
    Shared pre-work: parse TimeRange, build all masks, return components
    needed by both the single-plot and grid-plot functions.

    Returns
    -------
    dict with keys:
        BirthComboMask, DeathComboMask, AllComboMask,
        RangeStartMins, RangeEndMins, MinFrames
    """
    # ── Parse time range ───────────────────────────────────────────────────
    time_start_str, time_end_str = TimeRange.split('-')
    RangeStartHour = int(time_start_str.split(':')[0])
    RangeStartMin  = int(time_start_str.split(':')[1])
    RangeEndHour   = int(time_end_str.split(':')[0])
    RangeEndMin    = int(time_end_str.split(':')[1])
    RangeStartMins = RangeStartHour * 60 + RangeStartMin
    RangeEndMins   = RangeEndHour   * 60 + RangeEndMin

    # ── Duration filter ────────────────────────────────────────────────────
    MinFrames      = int(np.ceil(MinDuration / 5) + 1)

    # ── Build masks ────────────────────────────────────────────────────────
    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values

    TimeMask = (
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartMins) &
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute <  RangeEndMins)
    )
    BirthMask    = (FeatureFrames == 0)
    DeathMask    = (FeatureFrames == TrackDurations - 1)
    DurationMask = (TrackDurations >= MinFrames)

    return {
        'BirthComboMask' : TimeMask & DurationMask & BirthMask,
        'DeathComboMask' : TimeMask & DurationMask & DeathMask,
        'AllComboMask'   : TimeMask & DurationMask,
        'RangeStartMins' : RangeStartMins,
        'RangeEndMins'   : RangeEndMins,
        'MinFrames'      : MinFrames,
    }


def _BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max):
    """
    Add terrain shading, coastlines, and gridlines to an existing
    cartopy axes object.  Returns the axes (modified in place).
    """
    from matplotlib.colors     import ListedColormap, BoundaryNorm
    import matplotlib.ticker   as mticker
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    import cartopy.crs         as ccrs
    import cartopy.feature     as cfeature

    try:
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is None:
            raise ValueError('GEBCO DEM returned None')

        dem_lon  = dem_da.lon.values
        dem_lat  = dem_da.lat.values
        dem_data = dem_da.values

        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        if not np.any(np.isfinite(dem_data)):
            raise ValueError('DEM has no finite values in this domain')

        colours_topo = [
            '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
            '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
        ]
        bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
        cmap_elev   = ListedColormap(colours_topo)
        norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

        ax.pcolormesh(
            dem_lon_2d, dem_lat_2d, dem_data,
            cmap=cmap_elev, norm=norm_topo,
            alpha=1.0, transform=ccrs.PlateCarree(),
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[0.0], colors='black', linewidths=0.5,
            transform=ccrs.PlateCarree(), zorder=15,
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[400.0], colors='black', linewidths=0.3,
            transform=ccrs.PlateCarree(), zorder=15,
        )

    except Exception as e:
        print(f'  Terrain shading failed: {e}')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)
    gl_minor.ylocator = mticker.MultipleLocator(0.1)

    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)
    gl_mid.ylocator = mticker.MultipleLocator(0.5)

    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)
    gl_major.ylocator = mticker.MultipleLocator(1.0)

    for gl in (gl_mid, gl_major):
        gl.xformatter   = LONGITUDE_FORMATTER
        gl.yformatter   = LATITUDE_FORMATTER
        gl.top_labels   = False
        gl.right_labels = False

    return ax


def PlotFeatureLocationsSparse(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    TimeRange,
    DotsIncluded,
    MinDuration,
):
    """
    Plots feature locations (birth, death, or all frames) from a pre-loaded
    sparse PyFLEXTRKR xarray Dataset (SparseFXR).

    Produces two plots:
        1. Scatter plot  — small red dots on terrain background
        2. Density plot  — kernel density estimate on terrain background

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string (e.g. '22')
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD' inclusive date range
    TimeRange            : str        - 'HH:MM-HH:MM' UTC time-of-day window
    DotsIncluded         : str        - 'Birth', 'Death', or 'All'
    MinDuration          : float      - Minimum feature duration in minutes
    """

    import pandas as pd
    from scipy.stats           import gaussian_kde
    from pathlib               import Path
    import cartopy.crs         as ccrs
    import matplotlib.ticker   as mticker

    # ── Input validation ───────────────────────────────────────────────────
    if DotsIncluded not in ('Birth', 'Death', 'All'):
        raise ValueError("DotsIncluded must be 'Birth', 'Death', or 'All'.")

    # ── Parse date range ───────────────────────────────────────────────────
    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    # ── Build masks ────────────────────────────────────────────────────────
    masks     = _BuildFeatureMasks(SparseFXR, TimeRange, MinDuration)
    MinFrames = masks['MinFrames']

    if DotsIncluded == 'Birth':
        ActiveMask = masks['BirthComboMask']
    elif DotsIncluded == 'Death':
        ActiveMask = masks['DeathComboMask']
    else:
        ActiveMask = masks['AllComboMask']

    all_lats = SparseFXR['core_meanlat'].values[ActiveMask]
    all_lons = SparseFXR['core_meanlon'].values[ActiveMask]

    valid    = np.isfinite(all_lats) & np.isfinite(all_lons)
    all_lats = all_lats[valid]
    all_lons = all_lons[valid]

    print(f"Duration filter : >= {MinDuration} min  →  >= {MinFrames} frames")
    print(f"Date range      : {date_start.date()} to {date_end.date()}")
    print(f"Time window     : {TimeRange} UTC")
    print(f"Mode            : {DotsIncluded}")
    print(f"Points collected: {len(all_lats)}")

    if len(all_lats) == 0:
        print("No valid points found — nothing to plot.")
        return

    # ── Map extent ─────────────────────────────────────────────────────────
    lon_min = float(np.nanmin(all_lons))
    lon_max = float(np.nanmax(all_lons))
    lat_min = float(np.nanmin(all_lats))
    lat_max = float(np.nanmax(all_lats))

    # ── Radar location ─────────────────────────────────────────────────────
    _, LonShift     = GrabRadarInfo(RadarIDno)
    first_date_str  = DateRange[:8]
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_000000_QC2.nc'
    )
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {first_date_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')

    RadarXR_ref = xr.open_dataset(radar_file_path)
    RadarLon    = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat    = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')

    # ── Title / save ───────────────────────────────────────────────────────
    title_suffix = (
        f'{RadarSiteName} Radar  |  {DotsIncluded} Locations\n'
        f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
        f'{TimeRange} UTC  |  Min Duration: {MinDuration} min  |  '
        f'({len(all_lats)} points)'
    )

    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_{TimeRange.replace(":", "").replace("-", "_")}_'
        f'{DotsIncluded}_{int(MinDuration)}minMin'
    )

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 1 — Scatter
    # ══════════════════════════════════════════════════════════════════════

    print('\nGenerating scatter plot...')
    fig1, ax1 = plt.subplots(
        figsize    = (8, 6),
        subplot_kw = {'projection': ccrs.PlateCarree()},
        facecolor  = 'white',
    )
    _BuildBasemap(ax1, lon_min, lon_max, lat_min, lat_max)

    ax1.scatter(
        all_lons, all_lats,
        s=6, color='red', alpha=0.6,
        transform=ccrs.PlateCarree(), zorder=25, edgecolors='none',
    )
    ax1.plot(RadarLon, RadarLat, marker='*', color='black', markersize=8,
             transform=ccrs.PlateCarree(), zorder=30)
    ax1.plot(RadarLon, RadarLat, marker='*', color='white', markersize=4,
             transform=ccrs.PlateCarree(), zorder=31)

    ax1.set_title(f'Feature {title_suffix}', fontsize=10)
    plt.tight_layout()
    SavePath1 = SaveFolder + SaveBase + '_Scatter.png'
    # plt.savefig(SavePath1, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    print(f'  Scatter plot saved to: {SavePath1}')

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 2 — Kernel Density
    # ══════════════════════════════════════════════════════════════════════

    print('\nGenerating density plot...')
    fig2, ax2 = plt.subplots(
        figsize    = (8, 6),
        subplot_kw = {'projection': ccrs.PlateCarree()},
        facecolor  = 'white',
    )
    _BuildBasemap(ax2, lon_min, lon_max, lat_min, lat_max)

    xy             = np.vstack([all_lons, all_lats])
    kde            = gaussian_kde(xy, bw_method=0.1)
    lon_grid       = np.linspace(lon_min, lon_max, 300)
    lat_grid       = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    kde_values     = kde(np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])).reshape(lon_mesh.shape)

    mean_lat       = np.mean(all_lats)
    km2_per_deg2   = 111.32 * 111.32 * np.cos(np.radians(mean_lat))
    kde_per_100km2 = (kde_values / km2_per_deg2) * len(all_lats) * 100.0

    kde_max        = float(np.nanmax(kde_per_100km2))
    level_step     = kde_max / 25.0
    levels_fill    = np.arange(0, kde_max + level_step, level_step)
    levels_visible = levels_fill[1:]

    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels=levels_fill, cmap='jet', alpha=0.0,
        transform=ccrs.PlateCarree(), zorder=24, extend='max',
    )
    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels=levels_visible, cmap='jet', alpha=0.25,
        transform=ccrs.PlateCarree(), zorder=24, extend='max',
    )

    ax2.plot(RadarLon, RadarLat, marker='*', color='black', markersize=8,
             transform=ccrs.PlateCarree(), zorder=30)
    ax2.plot(RadarLon, RadarLat, marker='*', color='white', markersize=4,
             transform=ccrs.PlateCarree(), zorder=31)

    cax = fig2.add_axes([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])
    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=kde_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Locations per 100 km²', fontsize=9)
    tick_vals = np.linspace(0, kde_max, 11)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=8)

    ax2.set_title(f'Feature Density {title_suffix}', fontsize=10)
    plt.tight_layout()
    plt.draw()
    cax.set_position([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    SavePath2 = SaveFolder + SaveBase + '_Density.png'
    # plt.savefig(SavePath2, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Density plot saved to: {SavePath2}')


def PlotFeatureDensityGrid(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    MinDuration,
):
    """
    Produces three 4×2 grid figures of kernel density plots — one each for
    Birth, Death, and All — with one panel per 3-hour UTC block.

    All 8 panels within a figure share the same colour scale (normalised to
    the global maximum across all bins) and the same map extent (union of
    all points across all bins).

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD'
    MinDuration          : float      - Minimum feature duration in minutes
    """

    import pandas as pd
    from scipy.stats  import gaussian_kde
    from pathlib      import Path
    import cartopy.crs as ccrs

    # ── 3-hour time bins ───────────────────────────────────────────────────
    time_bins = [
        ('00:00', '03:00'),
        ('03:00', '06:00'),
        ('06:00', '09:00'),
        ('09:00', '12:00'),
        ('12:00', '15:00'),
        ('15:00', '18:00'),
        ('18:00', '21:00'),
        ('21:00', '24:00'),
    ]

    # ── Parse date range for titles ────────────────────────────────────────
    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    MinFrames = int(np.ceil(MinDuration / 5) + 1)

    # ── Radar location ─────────────────────────────────────────────────────
    _, LonShift     = GrabRadarInfo(RadarIDno)
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{date_start_str}/QC2/{RadarIDno}_{date_start_str}_000000_QC2.nc'
    )
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{date_start_str}/QC2/{RadarIDno}_{date_start_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {date_start_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')

    RadarXR_ref = xr.open_dataset(radar_file_path)
    RadarLon    = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat    = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    # print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')

    # ── Save folder ────────────────────────────────────────────────────────
    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_000000_240000_3hourblocks_{int(MinDuration)}minMin'
    )

    # ── Pre-compute base arrays used for masking ───────────────────────────
    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values
    DurationMask     = TrackDurations >= MinFrames
    BirthMask        = FeatureFrames == 0
    DeathMask        = FeatureFrames == (TrackDurations - 1)
    tod_mins         = BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute

    all_lats_raw = SparseFXR['core_meanlat'].values
    all_lons_raw = SparseFXR['core_meanlon'].values

    # ── Loop over the three modes ──────────────────────────────────────────
    for DotsIncluded in ('Birth', 'Death', 'All'):

        print(f'\n── Building density grid: {DotsIncluded} ──')

        if DotsIncluded == 'Birth':
            ModeMask = BirthMask
        elif DotsIncluded == 'Death':
            ModeMask = DeathMask
        else:
            ModeMask = np.ones(len(FeatureFrames), dtype=bool)

        # ── Pass 1: collect all points across all bins to get shared extent
        all_lats_combined = []
        all_lons_combined = []

        bin_lats = []
        bin_lons = []
        bin_counts = []

        for (t_start_str, t_end_str) in time_bins:
            t_start_mins = int(t_start_str.split(':')[0]) * 60 + int(t_start_str.split(':')[1])
            t_end_mins   = int(t_end_str.split(':')[0])   * 60 + int(t_end_str.split(':')[1])

            # handle 24:00 edge
            if t_end_mins == 1440:
                TimeMask = tod_mins >= t_start_mins
            else:
                TimeMask = (tod_mins >= t_start_mins) & (tod_mins < t_end_mins)

            ActiveMask = TimeMask & DurationMask & ModeMask

            lats = all_lats_raw[ActiveMask]
            lons = all_lons_raw[ActiveMask]
            valid = np.isfinite(lats) & np.isfinite(lons)
            lats  = lats[valid]
            lons  = lons[valid]

            bin_lats.append(lats)
            bin_lons.append(lons)
            bin_counts.append(len(lats))

            if len(lats) > 0:
                all_lats_combined.extend(lats)
                all_lons_combined.extend(lons)

            print(f'  {t_start_str}–{t_end_str} : {len(lats)} points')

        if len(all_lats_combined) == 0:
            print(f'  No points found for {DotsIncluded} — skipping.')
            continue

        all_lats_combined = np.array(all_lats_combined)
        all_lons_combined = np.array(all_lons_combined)

        # ── Shared map extent ──────────────────────────────────────────────
        lon_min = float(np.nanmin(all_lons_combined))
        lon_max = float(np.nanmax(all_lons_combined))
        lat_min = float(np.nanmin(all_lats_combined))
        lat_max = float(np.nanmax(all_lats_combined))

        # ── Shared KDE grid ────────────────────────────────────────────────
        lon_grid           = np.linspace(lon_min, lon_max, 300)
        lat_grid           = np.linspace(lat_min, lat_max, 300)
        lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
        grid_coords        = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])
        mean_lat           = np.mean(all_lats_combined)
        km2_per_deg2       = 111.32 * 111.32 * np.cos(np.radians(mean_lat))

        # ── Pass 2: compute KDE for each bin, find global max ──────────────
        kde_grids  = []
        global_max = 0.0

        for i, (lats, lons) in enumerate(zip(bin_lats, bin_lons)):
            if len(lats) < 2:
                kde_grids.append(None)
                continue

            kde        = gaussian_kde(np.vstack([lons, lats]), bw_method=0.1)
            kde_vals   = kde(grid_coords).reshape(lon_mesh.shape)
            kde_100km2 = (kde_vals / km2_per_deg2) * len(lats) * 100.0
            kde_grids.append(kde_100km2)
            global_max = max(global_max, float(np.nanmax(kde_100km2)))

        print(f'  Global KDE max : {global_max:.3f} per 100 km²')

        # ── Colour scale ───────────────────────────────────────────────────
        level_step     = global_max / 25.0
        levels_fill    = np.arange(0, global_max + level_step, level_step)
        levels_visible = levels_fill[1:]

        # ── Build 4×2 figure ───────────────────────────────────────────────
        fig, axes = plt.subplots(
            nrows      = 2,
            ncols      = 4,
            figsize    = (24, 12),
            subplot_kw = {'projection': ccrs.PlateCarree()},
            facecolor  = 'white',
        )
        fig.subplots_adjust(right=0.88, hspace=0.15, wspace=0.05)

        for panel_i, ax in enumerate(axes.flat):

            t_start_str, t_end_str = time_bins[panel_i]
            n_pts                  = bin_counts[panel_i]
            kde_grid               = kde_grids[panel_i]

            _BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max)

            if kde_grid is not None:
                # invisible contourf to anchor the shared colour scale
                ax.contourf(
                    lon_mesh, lat_mesh, kde_grid,
                    levels=levels_fill, cmap='jet', alpha=0.0,
                    transform=ccrs.PlateCarree(), zorder=24, extend='max',
                )
                # visible contourf
                ax.contourf(
                    lon_mesh, lat_mesh, kde_grid,
                    levels=levels_visible, cmap='jet', alpha=0.25,
                    transform=ccrs.PlateCarree(), zorder=24, extend='max',
                )

            # radar star
            ax.plot(RadarLon, RadarLat, marker='*', color='black',
                    markersize=8, transform=ccrs.PlateCarree(), zorder=30)
            ax.plot(RadarLon, RadarLat, marker='*', color='white',
                    markersize=4, transform=ccrs.PlateCarree(), zorder=31)

            ax.set_title(
                f'{t_start_str}–{t_end_str} UTC\n({n_pts} points)',
                fontsize=9,
            )

        # ── Shared colourbar ───────────────────────────────────────────────
        cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
        sm      = plt.cm.ScalarMappable(
            cmap = 'jet',
            norm = plt.Normalize(vmin=0, vmax=global_max),
        )
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='max')
        cbar.set_label('Feature Locations per 100 km²', fontsize=11)
        tick_vals = np.linspace(0, global_max, 11)
        cbar.set_ticks(tick_vals)
        cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
        cbar.ax.tick_params(labelsize=9)

        # ── Super title ────────────────────────────────────────────────────
        fig.suptitle(
            f'{RadarSiteName} Radar  |  {DotsIncluded} Feature Density  |  3-Hour UTC Blocks\n'
            f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
            f'Min Duration: {MinDuration} min',
            fontsize=13,
            y=0.98,
        )

        SavePath = SaveFolder + SaveBase + f'_{DotsIncluded}_DensityEx.png'
        plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=200)
        plt.close()
        print(f'  Saved: {SavePath}')


In [ ]:
PlotFeatureDensityGrid(
SparseFXR            = SparseFXR,
RadarIDno            = '22',
RadarSiteName        = 'Mackay',
QualityControlOption = 2,
DateRange            = '20240201-20240229',
MinDuration          = 30,
)
